In [2]:
# 0. Gerekli kütüphaneler
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
import os

In [3]:
# ---------- 1) Dosyayı oku ----------
dosyaYolu = r"C:\Users\omer\Desktop\cars.csv"  
df = pd.read_csv(dosyaYolu)
print("1) Orijinal veri boyutu:", df.shape)

1) Orijinal veri boyutu: (52256, 23)


In [4]:
# ---------- 2) Gereksiz sütunları kaldır ----------
drop_cols = [
    'baslik', 'konum', 'ilan_tarihi', 'renk', 'cekis',
    'arac_durumu', 'ortalama_yakit_tuketimi', 'yakit_deposu',
    'takasa_uygun', 'kimden', 'tramer'
]
# drop etmeden önce var mı kontrolü
drop_cols_existing = [c for c in drop_cols if c in df.columns]
df.drop(columns=drop_cols_existing, inplace=True)
print("2) Gereksiz sütunlar kaldırıldı. Kalan sütun sayısı:", df.shape[1])

2) Gereksiz sütunlar kaldırıldı. Kalan sütun sayısı: 12


In [5]:
# ---------- 3) Temel null kontrol ve bazı zorunlu sütunları temizle ----------
# Eğer model / vites_tipi / kasa_tipi gibi zorunlu sütunlar varsa null olanları sil
required_cols = ['model', 'vites_tipi', 'kasa_tipi', 'motor_hacmi', 'motor_gucu']
for col in required_cols:
    if col in df.columns:
        before = df.shape[0]
        df = df.dropna(subset=[col])
        after = df.shape[0]
        print(f"3) {col} için NaN satırlar silindi: {before-after} satır kaldırıldı")

3) model için NaN satırlar silindi: 71 satır kaldırıldı
3) vites_tipi için NaN satırlar silindi: 8 satır kaldırıldı
3) kasa_tipi için NaN satırlar silindi: 0 satır kaldırıldı
3) motor_hacmi için NaN satırlar silindi: 1134 satır kaldırıldı
3) motor_gucu için NaN satırlar silindi: 299 satır kaldırıldı


In [6]:
# ---------- 4) Fiyat sütununu temizle (örn. "1.250.000 TL" -> 1250000.0) ----------
if 'fiyat' in df.columns:
    # string ise işlem uygula, değilse atla
    df['fiyat'] = df['fiyat'].astype(str)
    # Remove non-digit except comma/dot then handle thousands separators
    # Adım: " TL" sil, harfleri kaldır, sonra nokta bin ayıracı olarak kaldıysa temizle, virgül ondalık mı diye kontrol et
    df['fiyat'] = df['fiyat'].str.replace(r'\s*TL\s*', '', regex=True)
    # Eğer nokta bin ayracı (örn 1.250.000) ise onları kaldıralım. Virgül ondalık varsa (nadiren) onu nokta yap
    # Temizleme: önce virgülü nokta yap (eğer ondalık gösteriyorsa), sonra bütün diğer noktaları kaldır
    # Fakat çoğu veride virgül ondalık yerine bin ayracı yok; güvenli yol:
    df['fiyat'] = df['fiyat'].str.replace(',', '', regex=False)   # virgülleri kaldır (bin ayracı varsa)
    df['fiyat'] = df['fiyat'].str.replace('.', '', regex=False)   # kalan noktaları da kaldır
    # Sonuçta sadece rakam kalmalı
    df['fiyat'] = pd.to_numeric(df['fiyat'], errors='coerce')
    print("4) Fiyat sütunu temizlendi. NaN sayısı:", df['fiyat'].isna().sum())


4) Fiyat sütunu temizlendi. NaN sayısı: 0


In [7]:
# ---------- 5) Kilometre sütununu temizle (örn. "124.000 km" -> 124000.0) ----------
if 'kilometre' in df.columns:
    df['kilometre'] = df['kilometre'].astype(str)
    df['kilometre'] = df['kilometre'].str.replace(r'\s*km\s*', '', regex=True)
    df['kilometre'] = df['kilometre'].str.replace(',', '', regex=False)
    df['kilometre'] = df['kilometre'].str.replace('.', '', regex=False)
    df['kilometre'] = pd.to_numeric(df['kilometre'], errors='coerce')
    print("5) Kilometre sütunu temizlendi. NaN sayısı:", df['kilometre'].isna().sum())

5) Kilometre sütunu temizlendi. NaN sayısı: 0


In [8]:
# ---------- 6) Mantıksız değerleri filtrele (sınırları ihtiyaca göre ayarla) ----------
# Örnek kurallar (senin önceki kodlara göre):
df = df[(df['fiyat'] >= 10000) & (df['fiyat'] <= 40_000_000)]
df = df[(df['kilometre'] >= 0) & (df['kilometre'] <= 700_000)]
# Özel hatalı kod: 111111 km varsa kaldır
if 'kilometre' in df.columns:
    df = df[df['kilometre'] != 111111]
print("6) Mantıksız kayıtlar çıkarıldı. Yeni boyut:", df.shape)

6) Mantıksız kayıtlar çıkarıldı. Yeni boyut: (50613, 12)


In [9]:
# ---------- 7) boya_degisen sütunundan degisen_sayisi ve boyali_sayisi çıkarma ----------
if 'boya_degisen' in df.columns:
    def parse_boya_degisen(s):
        if pd.isnull(s):
            return pd.Series([0, 0])
        s = str(s).lower()
        degisen = re.search(r'(\d+)\s*değişen', s)
        boyali = re.search(r'(\d+)\s*boyalı', s)
        degisen_sayisi = int(degisen.group(1)) if degisen else 0
        boyali_sayisi = int(boyali.group(1)) if boyali else 0
        return pd.Series([degisen_sayisi, boyali_sayisi])
    df[['degisen_sayisi', 'boyali_sayisi']] = df['boya_degisen'].apply(parse_boya_degisen)
    print("7) boya_degisen ayrıştırıldı.")

7) boya_degisen ayrıştırıldı.


In [10]:
# ---------- 8) motor_hacmi ve motor_gucu için düzgün numeric sütunlar oluştur ----------
# motor_hacmi örnekleri: "1.299 cc", "1300-1400 cc", "1300,5" vb - önce string temizle, sonra ortalama al
def ort_motor_hacmi_duzgun(s):
    if pd.isnull(s):
        return np.nan
    s = str(s)
    # ondalık virgül varsa noktaya çevir
    s = s.replace(',', '.')
    # eğer "-" range ise ortalamasını al
    if '-' in s:
        parts = re.split(r'[-–—]', s)
        nums = []
        for p in parts[:2]:
            m = re.search(r'(\d+(\.\d+)?)', p)
            if m:
                nums.append(float(m.group(1)))
        if len(nums) == 2:
            return np.mean(nums)
    # değilse ilk sayıyı al
    m = re.search(r'(\d+(\.\d+)?)', s)
    return float(m.group(1)) if m else np.nan

def ort_motor_gucu_duzgun(s):
    if pd.isnull(s):
        return np.nan
    s = str(s)
    s = s.replace(',', '.')
    if '-' in s:
        parts = re.split(r'[-–—]', s)
        nums = []
        for p in parts[:2]:
            m = re.search(r'(\d+(\.\d+)?)', p)
            if m:
                nums.append(float(m.group(1)))
        if len(nums) == 2:
            return np.mean(nums)
    m = re.search(r'(\d+(\.\d+)?)', s)
    return float(m.group(1)) if m else np.nan

if 'motor_hacmi' in df.columns:
    df['motor_hacmi_num'] = df['motor_hacmi'].apply(ort_motor_hacmi_duzgun)
if 'motor_gucu' in df.columns:
    df['motor_gucu_num'] = df['motor_gucu'].apply(ort_motor_gucu_duzgun)

print("8) motor_hacmi_num ve motor_gucu_num oluşturuldu. NaN sayıları:",
      df.get('motor_hacmi_num').isna().sum() if 'motor_hacmi_num' in df.columns else 'yok',
      df.get('motor_gucu_num').isna().sum() if 'motor_gucu_num' in df.columns else 'yok')

8) motor_hacmi_num ve motor_gucu_num oluşturuldu. NaN sayıları: 139 117


In [ ]:
# ---------- 9) Orjinal uzun sütunları kaldır ----------
for c in ['boya_degisen', 'motor_hacmi', 'motor_gucu']:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)


In [12]:
# ---------- 10) '-' veya eksikleri temizleme ve kasa_tipi doldurma ----------
if 'kasa_tipi' in df.columns:
    df['kasa_tipi'] = df['kasa_tipi'].replace('-', pd.NA)
    # Marka-seri-model gruplarına göre doldurma
    group_cols = [c for c in ['marka','seri','model'] if c in df.columns]
    if group_cols:
        df['kasa_tipi'] = df.groupby(group_cols)['kasa_tipi'].transform(lambda x: x.ffill().bfill())
    # hala boşsa en sık olan ile doldur
    if df['kasa_tipi'].isna().any():
        try:
            en_sik = df['kasa_tipi'].mode().iloc[0]
            df['kasa_tipi'] = df['kasa_tipi'].fillna(en_sik)
        except Exception:
            pass
    print("10) kasa_tipi temizlendi / dolduruldu. NaN sayısı:", df['kasa_tipi'].isna().sum())

10) kasa_tipi temizlendi / dolduruldu. NaN sayısı: 0


In [13]:
# ---------- 11) Gerekliyse motor_hacmi_num / motor_gucu_num içindeki ',' -> '.' düzeltmesi ----------
# (Bu kod zaten float üretti, ama yine de güvenlik)
for col in ['motor_hacmi_num', 'motor_gucu_num']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [14]:
# ---------- 12) Nihai NaN temizliği: motor değerleri kritikse bunları bırakma ----------
# (Senin daha önce yaptığın gibi) motor_hacmi_num veya motor_gucu_num boşsa sil
for must in ['motor_hacmi_num', 'motor_gucu_num']:
    if must in df.columns:
        before = df.shape[0]
        df = df.dropna(subset=[must])
        after = df.shape[0]
        print(f"12) {must} NaN satırları silindi: {before-after} kaldırıldı")

print("12) Temizleme sonrası veri boyutu:", df.shape)

12) motor_hacmi_num NaN satırları silindi: 139 kaldırıldı
12) motor_gucu_num NaN satırları silindi: 94 kaldırıldı
12) Temizleme sonrası veri boyutu: (50380, 13)


In [15]:
# ---------- 13) Kategorik dönüşümler: LabelEncode marka/seri/model ve One-Hot vites/yakit/kasa ----------
label_cols = [c for c in ['marka','seri','model'] if c in df.columns]
le_dict = {}
for col in label_cols:
    le = LabelEncoder()
    df[col + '_num'] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le
print("13) Label Encoding tamamlandı:", [c + '_num' for c in label_cols])

onehot_cols = [c for c in ['vites_tipi','yakit_tipi','kasa_tipi'] if c in df.columns]
df = pd.get_dummies(df, columns=onehot_cols, drop_first=False)
print("13) One-Hot Encoding tamamlandı. Toplam sütun sayısı:", df.shape[1])

13) Label Encoding tamamlandı: ['marka_num', 'seri_num', 'model_num']
13) One-Hot Encoding tamamlandı. Toplam sütun sayısı: 31


In [16]:
# ---------- 14) Son kontroller: sütun isimleri, veri tipleri, eksik yüzdeleri ----------
print("\n14) Son hal - şekil:", df.shape)
print(df.info())
print("\nEksik yüzdeleri (%):\n", (df.isnull().sum()/len(df)*100).round(3))


14) Son hal - şekil: (50380, 31)
<class 'pandas.core.frame.DataFrame'>
Index: 50380 entries, 0 to 52255
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   fiyat                     50380 non-null  int64  
 1   marka                     50380 non-null  object 
 2   seri                      50380 non-null  object 
 3   model                     50380 non-null  object 
 4   yil                       50380 non-null  int64  
 5   kilometre                 50380 non-null  int64  
 6   degisen_sayisi            50380 non-null  int64  
 7   boyali_sayisi             50380 non-null  int64  
 8   motor_hacmi_num           50380 non-null  float64
 9   motor_gucu_num            50380 non-null  float64
 10  marka_num                 50380 non-null  int64  
 11  seri_num                  50380 non-null  int64  
 12  model_num                 50380 non-null  int64  
 13  vites_tipi_Düz            50380 

In [17]:
# ---------- 15) Son olarak dosyayı masaüstüne CSV olarak kaydet ----------
desktop_path = r"C:\Users\omer\Desktop"
out_name = "arac_fiyat_tahmin_veri_seti_v1.csv"
out_path = os.path.join(desktop_path, out_name)

df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\n15) Temizlenmiş ve encode edilmiş CSV dosyası kaydedildi: {out_path}")

# Opsiyonel: Encoded excel de kaydet
excel_out = os.path.join(desktop_path, "arac_fiyat_tahmin_veri_seti_v1_encoded.xlsx")
df.to_excel(excel_out, index=False)
print(f"    Encoded Excel kaydedildi: {excel_out}")


15) Temizlenmiş ve encode edilmiş CSV dosyası kaydedildi: C:\Users\omer\Desktop\arac_fiyat_tahmin_veri_seti_v1.csv
    Encoded Excel kaydedildi: C:\Users\omer\Desktop\arac_fiyat_tahmin_veri_seti_v1_encoded.xlsx


In [18]:
# ---------- 16) Kısa özet çıktı ----------
print("\n✅ İşlem tamam. Son veri boyutu:", df.shape)
print("✅ CSV masaüstüne kaydedildi:", out_path)



✅ İşlem tamam. Son veri boyutu: (50380, 31)
✅ CSV masaüstüne kaydedildi: C:\Users\omer\Desktop\arac_fiyat_tahmin_veri_seti_v1.csv


In [26]:
import pandas as pd
import os

# Masaüstü yolu
desktop_path = r"C:\Users\omer\Desktop"

# LabelEncoder mappinglerini DataFrame'e çeviren fonksiyon
def le_to_df(le, col_name):
    return pd.DataFrame({
        col_name + '_num': range(len(le.classes_)),
        col_name: le.inverse_transform(range(len(le.classes_)))
    })

# Her sütun için DataFrame oluştur
marka_df = le_to_df(le_dict['marka'], 'marka')
seri_df = le_to_df(le_dict['seri'], 'seri')
model_df = le_to_df(le_dict['model'], 'model')

# Dosya yolları
marka_path = os.path.join(desktop_path, "marka.xlsx")
seri_path = os.path.join(desktop_path, "seri.xlsx")
model_path = os.path.join(desktop_path, "model.xlsx")

# Excel olarak kaydet (her biri ayrı dosya)
marka_df.to_excel(marka_path, index=False)
seri_df.to_excel(seri_path, index=False)
model_df.to_excel(model_path, index=False)

print("✅ Tüm LabelEncoder eşleşmeleri ayrı ayrı Excel dosyalarına kaydedildi:")
print("Marka:", marka_path)
print("Seri:", seri_path)
print("Model:", model_path)


✅ Tüm LabelEncoder eşleşmeleri ayrı ayrı Excel dosyalarına kaydedildi:
Marka: C:\Users\omer\Desktop\marka.xlsx
Seri: C:\Users\omer\Desktop\seri.xlsx
Model: C:\Users\omer\Desktop\model.xlsx


In [29]:
import pandas as pd
import os

desktop_path = r"C:\Users\omer\Desktop"

# 1️⃣ Marka tablosu
marka_df = pd.DataFrame({
    'marka_id': range(len(le_dict['marka'].classes_)),
    'marka_adı': le_dict['marka'].inverse_transform(range(len(le_dict['marka'].classes_)))
})

# 2️⃣ Seri tablosu
# Doğru sütun isimleri: 'marka_num', 'seri_num', 'seri'
seri_unique = df[['marka_num','seri_num','seri']].drop_duplicates()
seri_df = seri_unique.rename(columns={
    'marka_num': 'marka_id',
    'seri_num': 'seri_id',
    'seri': 'seri_adı'
})
seri_df = seri_df[['seri_id','marka_id','seri_adı']].drop_duplicates()

# 3️⃣ Model tablosu
# Doğru sütunlar: 'seri_num','model_num','model'
model_unique = df[['seri_num','model_num','model']].drop_duplicates()
model_df = model_unique.rename(columns={
    'seri_num': 'seri_id',
    'model_num': 'model_id',
    'model': 'model_adı'
})
model_df = model_df[['model_id','seri_id','model_adı']].drop_duplicates()

# Excel olarak kaydet
marka_df.to_excel(os.path.join(desktop_path,"marka.xlsx"), index=False)
seri_df.to_excel(os.path.join(desktop_path,"seri.xlsx"), index=False)
model_df.to_excel(os.path.join(desktop_path,"model.xlsx"), index=False)

print("✅ Marka, Seri, Model tabloları ayrı ayrı Excel dosyalarına kaydedildi.")


✅ Marka, Seri, Model tabloları ayrı ayrı Excel dosyalarına kaydedildi.
